# Working with LLM APIs in Python

**Author:** Manish Kundu

A compact notebook demonstrating how to call language models through:
- the OpenAI API
- Google's OpenAI-compatible Gemini endpoint
- a local Ollama server

The examples use environment variables for credentials and keep each provider optional.

## Setup

Install the dependencies:

```bash
pip install openai requests python-dotenv
```

Create a `.env` file in the project directory and add the keys you plan to use:

```dotenv
OPENAI_API_KEY=your_openai_api_key
GOOGLE_API_KEY=your_google_api_key
```

Do not commit `.env` or API keys to GitHub. Add `.env` to `.gitignore`.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("OpenAI key configured:", bool(OPENAI_API_KEY))
print("Google key configured:", bool(GOOGLE_API_KEY))

## 1. Call the OpenAI Chat Completions API

The `requests` example sends an HTTP request directly. Set `OPENAI_API_KEY` in `.env` before running it.

In [ ]:
if not OPENAI_API_KEY:
    print("Skipping OpenAI example: set OPENAI_API_KEY in .env.")
else:
    response = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": "gpt-4.1-mini",
            "messages": [{"role": "user", "content": "Share one interesting science fact."}],
        },
        timeout=60,
    )
    response.raise_for_status()
    print(response.json()["choices"][0]["message"]["content"])

## 2. Use the OpenAI Python SDK

The SDK wraps the HTTP request in a Python interface.

In [ ]:
if not OPENAI_API_KEY:
    print("Skipping SDK example: set OPENAI_API_KEY in .env.")
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": "Explain an API in one sentence."}],
    )
    print(response.choices[0].message.content)

## 3. Call Gemini through an OpenAI-compatible endpoint

This uses the OpenAI Python SDK with Google's compatible base URL. Set `GOOGLE_API_KEY` in `.env` to run this section.

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

if not GOOGLE_API_KEY:
    print("Skipping Gemini example: set GOOGLE_API_KEY in .env.")
else:
    gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GOOGLE_API_KEY)
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Explain embeddings in one sentence."}],
    )
    print(response.choices[0].message.content)

## 4. Run a model locally with Ollama

Install Ollama from [ollama.com](https://ollama.com), start the local service, and download a model:

```bash
ollama serve
ollama pull llama3.2
```

Ollama exposes an OpenAI-compatible API on `http://localhost:11434/v1`.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

try:
    health = requests.get("http://localhost:11434", timeout=3)
    health.raise_for_status()
    print("Ollama is running.")
except requests.RequestException:
    print("Ollama is not reachable. Start it with `ollama serve`.")

In [ ]:
ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",  # Ollama runs locally; this value is not a secret.
)

try:
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[{"role": "user", "content": "Share one interesting science fact."}],
    )
    print(response.choices[0].message.content)
except Exception as exc:
    print("Could not call Ollama. Check that the service is running and llama3.2 is installed.")
    print(f"Details: {exc}")

## Key takeaway

The same chat-completions pattern can work across hosted APIs and local models. Keep credentials in environment variables, check API errors, and never commit secrets to source control.